# Reference-Metric Applications

Use reference metrics to inspect coordinate maps, scale factors, determinant
identities, and precomputed geometry factors.

Navigation: [Index][index] | Previous: [Cartesian Wave Project][prev]
| Next: [Basis Transforms][next]

[index]: ../index.ipynb
[prev]: ../3-wave_equation/start_to_finish_wave_cartesian.ipynb
[next]: basis_transforms.ipynb

## Learning Goals

- Use reference metrics to organize coordinate geometry.
- Check determinant and scale-factor identities.
- See how precomputed geometry data supports generated curvilinear code.

## Words for This Notebook

- **Reference metric:** the built-in geometry of a coordinate system.
- **Jacobian:** a table of derivatives that converts between coordinate
  labels.
- **Scale factor:** a coordinate-dependent length factor for one coordinate
  direction.
- **Determinant:** one number built from a matrix that often measures volume
  scaling.
- **Precompute:** calculate reusable coordinate quantities once instead of
  repeatedly.

Use the code cells actively: predict the result, run the cell, then explain
the output in plain language.

## Notation

In orthogonal coordinates, the reference metric diagonal is built from scale
factors:

$$
\hat{\gamma}_{ii} = h_i^2.
$$

The determinant identity checked below is

$$
\det(\hat{\gamma}) = (h_0 h_1 h_2)^2.
$$

## Step 1: Import Tools

These imports provide SymPy and the NRPy modules used below.

In [1]:
import sympy as sp

import nrpy.params as par
import nrpy.reference_metric as refmetric
from nrpy.infrastructures.BHaH.rfm_precompute import ReferenceMetricPrecompute

## Step 2: Compare Coordinate-System Catalog Entries

| Coordinate system | Purpose | What to inspect | Validation target |
| --- | --- | --- | --- |
| `Cartesian` | baseline flat coordinates | unit scale factors | determinant residual |
| `Spherical` | polar-angle geometry | `r` and `sin(theta)` factors | determinant residual |
| `SinhSpherical` | radially stretched sphere | sinh radial factor | determinant residual |
| `SinhCylindrical` | stretched cylinder | radial and vertical factors | determinant residual |

In [2]:
coord_systems = ["Cartesian", "Spherical", "SinhSpherical", "SinhCylindrical"]
catalog_rows = []
print("coord_system | xx | scale_factors | detgammahat | radial_like_dirs")
for coord_system in coord_systems:
    metric = refmetric.ReferenceMetric(coord_system, enable_rfm_precompute=False)
    catalog_rows.append((coord_system, metric))
    print(
        coord_system,
        "|",
        metric.xx,
        "|",
        metric.scalefactor_orthog,
        "|",
        metric.detgammahat,
        "|",
        metric.radial_like_dirns,
    )
    print("  Cartesian map:", metric.xx_to_Cart)
    print("  ghatDD diagonal:", [metric.ghatDD[i][i] for i in range(3)])
    print("  ghatUU diagonal:", [metric.ghatUU[i][i] for i in range(3)])

coord_system | xx | scale_factors | detgammahat | radial_like_dirs
Cartesian | [xx0, xx1, xx2] | [1, 1, 1] | 1 | [0, 1, 2]
  Cartesian map: [xx0, xx1, xx2]
  ghatDD diagonal: [1, 1, 1]
  ghatUU diagonal: [1, 1, 1]
Spherical | [xx0, xx1, xx2] | [1, xx0, xx0*sin(xx1)] | xx0**4*sin(xx1)**2 | [0]
  Cartesian map: [xx0*sin(xx1)*cos(xx2), xx0*sin(xx1)*sin(xx2), xx0*cos(xx1)]
  ghatDD diagonal: [1, xx0**2, xx0**2*sin(xx1)**2]
  ghatUU diagonal: [1, xx0**(-2), 1/(xx0**2*sin(xx1)**2)]


SinhSpherical | [xx0, xx1, xx2] | [AMPL*(exp(xx0/SINHW)/SINHW + exp(-xx0/SINHW)/SINHW)/(exp(1/SINHW) - exp(-1/SINHW)), AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))/(exp(1/SINHW) - exp(-1/SINHW)), AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))*sin(xx1)/(exp(1/SINHW) - exp(-1/SINHW))] | AMPL**6*(exp(xx0/SINHW)/SINHW + exp(-xx0/SINHW)/SINHW)**2*(exp(xx0/SINHW) - exp(-xx0/SINHW))**4*sin(xx1)**2/(exp(1/SINHW) - exp(-1/SINHW))**6 | [0]
  Cartesian map: [AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))*sin(xx1)*cos(xx2)/(exp(1/SINHW) - exp(-1/SINHW)), AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))*sin(xx1)*sin(xx2)/(exp(1/SINHW) - exp(-1/SINHW)), AMPL*(exp(xx0/SINHW) - exp(-xx0/SINHW))*cos(xx1)/(exp(1/SINHW) - exp(-1/SINHW))]
  ghatDD diagonal: [AMPL**2*(exp(xx0/SINHW)/SINHW + exp(-xx0/SINHW)/SINHW)**2/(exp(1/SINHW) - exp(-1/SINHW))**2, AMPL**2*(exp(xx0/SINHW) - exp(-xx0/SINHW))**2/(exp(1/SINHW) - exp(-1/SINHW))**2, AMPL**2*(exp(xx0/SINHW) - exp(-xx0/SINHW))**2*sin(xx1)**2/(exp(1/SINHW) - exp(-1/SINHW))**2]
  ghatUU dia

## Validation Check

A zero residual confirms that each determinant equals the square of its
scale-factor product.

In [3]:
determinant_residuals = {}
for coord_system, metric in catalog_rows:
    scale_product = sp.prod(metric.scalefactor_orthog)
    residual = sp.trigsimp(sp.simplify(metric.detgammahat - scale_product**2))
    determinant_residuals[coord_system] = residual
print("coord_system | determinant residual")
for coord_system, residual in determinant_residuals.items():
    print(coord_system, "|", residual)
failed_residuals = {
    coord_system: residual
    for coord_system, residual in determinant_residuals.items()
    if residual != 0
}
if failed_residuals:
    raise RuntimeError(f"Nonzero determinant residuals: {failed_residuals}")

coord_system | determinant residual
Cartesian | 0
Spherical | 0
SinhSpherical | 0
SinhCylindrical | 0


## Step 3: Register Precompute Support Data

Named parameters keep numerical choices separate from symbolic formulas. This
cell restores the runtime settings it changes after the dynamic member list is
printed.

| Member pattern | Meaning |
| --- | --- |
| `f0_of_xx0` | reusable function of the first coordinate |
| `f0_of_xx0__D0` | first derivative with respect to the first coordinate |
| `f3_of_xx2` | reusable function of the third coordinate |
| `f3_of_xx2__DD22` | second derivative with respect to the third coordinate |

In [4]:
saved_runtime_settings = {
    name: par.parval_from_str(name)
    for name in [
        "parallelization",
        "fp_type",
        "CoordSystem_to_register_CodeParameters",
    ]
}
try:
    par.set_parval_from_str("parallelization", "openmp")
    par.set_parval_from_str("fp_type", "double")
    par.set_parval_from_str(
        "CoordSystem_to_register_CodeParameters", "SinhCylindrical"
    )
    precompute_metric = refmetric.reference_metric["SinhCylindrical_rfm_precompute"]
    precompute = ReferenceMetricPrecompute("SinhCylindrical")
    print("precompute metric:", precompute_metric.CoordSystem)
    print("stored member count:", len(precompute.member_specs))
    print("member | allocation")
    for name, declaration in precompute.member_specs:
        print(name, "|", declaration)
finally:
    for name, value in saved_runtime_settings.items():
        par.set_parval_from_str(name, value)
print("restored runtime settings:", sorted(saved_runtime_settings))

Setting up reference_metric[SinhCylindrical_rfm_precompute]...


precompute metric: SinhCylindrical
stored member count: 7
member | allocation
f0_of_xx0 | (size_t)params->Nxx_plus_2NGHOSTS0
f0_of_xx0__D0 | (size_t)params->Nxx_plus_2NGHOSTS0
f0_of_xx0__DD00 | (size_t)params->Nxx_plus_2NGHOSTS0
f0_of_xx0__DDD000 | (size_t)params->Nxx_plus_2NGHOSTS0
f3_of_xx2 | (size_t)params->Nxx_plus_2NGHOSTS2
f3_of_xx2__D2 | (size_t)params->Nxx_plus_2NGHOSTS2
f3_of_xx2__DD22 | (size_t)params->Nxx_plus_2NGHOSTS2
restored runtime settings: ['CoordSystem_to_register_CodeParameters', 'fp_type', 'parallelization']


The printed Jacobian data and metric factors show how coordinate geometry
enters a differential equation. Curvilinear wave update routines use these
factors to keep the physics independent of the chosen coordinates.

## Learning Check

Before checking the determinant identity, predict why spherical coordinates
need a volume factor involving `r` and `sin(theta)`.

## Continue to Basis Transforms

- [Reference Metrics](../1-intro/reference_metric.ipynb)
- [Basis Transforms](basis_transforms.ipynb)
- [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb)